In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, roc_auc_score
from pgmpy.models import BayesianNetwork
from pgmpy.estimators import HillClimbSearch, BicScore
from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from skrebate import ReliefF  # Ensure skrebate is installed
from joblib import Parallel, delayed
import warnings
import multiprocessing

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Load the dataset
data = pd.read_excel("class123_dataset.xlsx")

# Extract predictors (X) and outcome (Y)
X = data.drop('RRI', axis=1)
Y = data['RRI']

# === Step 1: Discretize all features to binary based on global median ===

def discretize_binary_global(X_df):
    """
    Discretize continuous features into binary based on the global median.

    Parameters:
    - X_df: DataFrame containing feature columns.

    Returns:
    - Discretized DataFrame with binary features.
    """
    X_discretized = X_df.copy()
    for column in X_discretized.columns:
        median = X_discretized[column].median()
        X_discretized[column] = (X_discretized[column] > median).astype(int)
    return X_discretized

# Apply global discretization
X_discretized = discretize_binary_global(X)

# Combine discretized features with the outcome
data_discretized = X_discretized.copy()
data_discretized['RRI'] = Y

# Initialize 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Define a function to evaluate a single fold for a given k
def evaluate_fold(k, sorted_features, fold, train_index, test_index):
    # Split data into training and testing sets
    train_data = data_discretized.iloc[train_index].copy()
    test_data = data_discretized.iloc[test_index].copy()
    
    # Separate predictors and outcome
    X_train = train_data.drop('RRI', axis=1)
    Y_train = train_data['RRI']
    X_test = test_data.drop('RRI', axis=1)
    Y_test = test_data['RRI']
    
    # Apply Relief on training data
    relief_fold = ReliefF(n_neighbors=100, n_features_to_select='all')
    relief_fold.fit(X_train.values, Y_train.values)
    feature_scores_fold = relief_fold.feature_importances_
    feature_ranking_fold = np.argsort(feature_scores_fold)[::-1]
    sorted_features_fold = X_train.columns[feature_ranking_fold]
    
    # Select top k features based on the current fold's ranking
    selected_features = sorted_features_fold[:k]
    
    # Prepare training and testing data with selected features
    train_selected = train_data[selected_features.tolist() + ['RRI']]
    test_selected = test_data[selected_features.tolist() + ['RRI']]
    
    # Learn the Bayesian Network structure using training data
    hc = HillClimbSearch(train_selected)
    try:
        best_model_structure = hc.estimate(scoring_method=BicScore(train_selected))
    except Exception as e:
        print(f"    HillClimbSearch failed on fold {fold} for k={k}: {e}")
        return None  # Skip this fold if structure learning fails
    
    # Create and fit the Bayesian Network model
    model = BayesianNetwork(best_model_structure.edges())
    try:
        model.fit(train_selected, estimator=BayesianEstimator)
    except Exception as e:
        print(f"    BayesianEstimator failed on fold {fold} for k={k}: {e}")
        return None  # Skip this fold if model fitting fails
    
    # Perform inference
    infer = VariableElimination(model)
    
    # Predict probabilities for the test set
    y_true = test_selected['RRI']
    y_pred_probs = []
    
    network_vars = set(model.nodes())
    
    for _, row in test_selected.iterrows():
        evidence = {col: row[col] for col in selected_features if col in network_vars}
        try:
            result = infer.query(variables=['RRI'], evidence=evidence, show_progress=False)
            # Assuming 'RRI' has states [0, 1]
            y_pred_probs.append(result.values[1])  # Probability of class 1 (positive class)
        except Exception as e:
            print(f"    Error during inference on fold {fold} for k={k}: {e}")
            y_pred_probs.append(0)  # Assign a default probability or handle appropriately
    
    # Convert probabilities to binary predictions
    y_pred = [1 if prob > 0.5 else 0 for prob in y_pred_probs]
    
    # Calculate performance metrics
    try:
        auc = roc_auc_score(y_true, y_pred_probs)
    except ValueError:
        auc = 0.5  # Assign a default AUC if only one class is present
    accuracy = accuracy_score(y_true, y_pred)
    
    return {'auc': auc, 'accuracy': accuracy}

# Define a function to evaluate a single k across all folds
def evaluate_k(k, sorted_features):
    print(f"\nEvaluating top {k} feature(s): {list(sorted_features[:k])}")
    
    # Initialize a list to store fold results
    fold_results = []
    
    # Process each fold sequentially
    for fold, (train_index, test_index) in enumerate(kf.split(data_discretized), 1):
        result = evaluate_fold(k, sorted_features, fold, train_index, test_index)
        if result is not None:
            fold_results.append(result)
    
    # Calculate average AUC and accuracy
    if fold_results:
        auc_scores = [res['auc'] for res in fold_results]
        accuracy_scores = [res['accuracy'] for res in fold_results]
        avg_auc = np.mean(auc_scores)
        std_auc = np.std(auc_scores)
        avg_accuracy = np.mean(accuracy_scores)
        std_accuracy = np.std(accuracy_scores)
    else:
        avg_auc = 0
        std_auc = 0
        avg_accuracy = 0
        std_accuracy = 0
    
    print(f"  Average AUC for top {k} feature(s): {avg_auc:.4f} ± {std_auc:.4f}")
    print(f"  Average Accuracy for top {k} feature(s): {avg_accuracy:.4f} ± {std_accuracy:.4f}")
    
    return {'k': k, 'avg_auc': avg_auc, 'std_auc': std_auc, 
            'avg_accuracy': avg_accuracy, 'std_accuracy': std_accuracy}

# === Step 2: Feature Ranking using ReliefF on Discretized Features ===

# Initialize Relief for feature ranking
relief = ReliefF(n_neighbors=100, n_features_to_select='all')  # Adjust n_neighbors as needed
relief.fit(X_discretized.values, Y.values)
feature_scores = relief.feature_importances_
feature_ranking = np.argsort(feature_scores)[::-1]  # Indices of features sorted by importance
sorted_features = X_discretized.columns[feature_ranking]

# Limit to top 50 features
top_k_limit = 40
num_features = min(top_k_limit, X_discretized.shape[1])  # Ensure we don't exceed available features
sorted_features = sorted_features[:num_features]

print("Feature ranking based on Relief (top 50):")
for rank, feature in enumerate(sorted_features, start=1):
    print(f"{rank}. {feature} (Score: {feature_scores[feature_ranking[rank-1]]:.4f})")

# Define the range of k (number of features to select) from 1 to 50
k_values = range(1, num_features + 1)

# Initialize dictionaries to store AUC and Accuracy scores
auc_scores_dict = {}
accuracy_scores_dict = {}

# === Step 3: Evaluate Different Feature Subset Sizes in Parallel ===

print("\nStarting evaluation of top 50 features across different subset sizes...")

# Set up parallel processing for different k values
# Each k is independent, so they can be processed in parallel
# To avoid excessive memory usage, you might limit the number of parallel jobs
# For example, use n_jobs= multiprocessing.cpu_count() // 2 or similar
# Here, we use all available cores
results_k = Parallel(n_jobs=-1, verbose=10)(
    delayed(evaluate_k)(k, sorted_features) for k in k_values
)

# Process and store the results
for res in results_k:
    k = res['k']
    auc_scores_dict[k] = (res['avg_auc'], res['std_auc'])
    accuracy_scores_dict[k] = (res['avg_accuracy'], res['std_accuracy'])

# Identify the feature subset with the highest average AUC
best_k = max(auc_scores_dict, key=lambda k: auc_scores_dict[k][0])
best_auc, best_auc_std = auc_scores_dict[best_k]
best_features = sorted_features[:best_k]

print("\n=== Feature Selection Results ===")
print(f"Best number of features: {best_k}")
print(f"Best feature subset: {list(best_features)}")
print(f"Best Average AUC: {best_auc:.4f} ± {best_auc_std:.4f}")

/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


Feature ranking based on Relief (top 50):
1. rs911263 (Score: 0.2158)
2. navicular_drop_asymmetry (Score: 0.2144)
3. rs7035322 (Score: 0.2135)
4. Duty_factor_asymmetry_10 (Score: 0.2071)
5. rs4454832 (Score: 0.2061)
6. knee_flexion_peak_torque_asymmetry (Score: 0.2043)
7. rs1590 (Score: 0.2035)
8. rs1800797 (Score: 0.2021)
9. rs3753841 (Score: 0.2015)
10. rs17756404 (Score: 0.2008)
11. past_stress_injury (Score: 0.2000)
12. rs10759753 (Score: 0.1979)
13. rs3789870 (Score: 0.1973)
14. hip_adduction_peak_torque_asymmetry (Score: 0.1968)
15. rs143383 (Score: 0.1966)
16. Duty_factor_asymmetry_12 (Score: 0.1956)
17. Impact_peak_asymmetry_10 (Score: 0.1956)
18. Athlete_Score (Score: 0.1955)
19. Flight_time_10 (Score: 0.1953)
20. hip_abduction_peak_torque (Score: 0.1947)
21. rs2289360 (Score: 0.1940)
22. rs2252070 (Score: 0.1939)
23. total_ad_ab_ratio (Score: 0.1931)
24. knee_extension_peak_angle (Score: 0.1927)
25. ad_ab_ratio_asymmetry (Score: 0.1927)
26. EDEQ_total (Score: 0.1922)
27. rs18

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
/home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
  0%|          | 0/1000000 [00:00<?, ?it/s]


Evaluating top 10 feature(s): ['rs911263', 'navicular_drop_asymmetry', 'rs7035322', 'Duty_factor_asymmetry_10', 'rs4454832', 'knee_flexion_peak_torque_asymmetry', 'rs1590', 'rs1800797', 'rs3753841', 'rs17756404']


  0%|          | 44/1000000 [00:01<7:20:20, 37.85it/s] /home/h_wu4/ml_env/lib/python3.8/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
  0%|          | 2/1000000 [00:00<6:14:09, 44.54it/s]

...

  Average AUC for top 39 feature(s): 0.6113 ± 0.0362
  Average Accuracy for top 39 feature(s): 0.9088 ± 0.0102



  0%|          | 250/1000000 [06:37<6477:06:27, 23.32s/it] 

  Average AUC for top 38 feature(s): 0.6064 ± 0.0273
  Average Accuracy for top 38 feature(s): 0.9088 ± 0.0102



  0%|          | 261/1000000 [20:35<1314:39:46,  4.73s/it] [Parallel(n_jobs=-1)]: Done  40 out of  40 | elapsed: 372.0min finished



=== Feature Selection Results ===
Best number of features: 33
Best feature subset: ['rs911263', 'navicular_drop_asymmetry', 'rs7035322', 'Duty_factor_asymmetry_10', 'rs4454832', 'knee_flexion_peak_torque_asymmetry', 'rs1590', 'rs1800797', 'rs3753841', 'rs17756404', 'past_stress_injury', 'rs10759753', 'rs3789870', 'hip_adduction_peak_torque_asymmetry', 'rs143383', 'Duty_factor_asymmetry_12', 'Impact_peak_asymmetry_10', 'Athlete_Score', 'Flight_time_10', 'hip_abduction_peak_torque', 'rs2289360', 'rs2252070', 'total_ad_ab_ratio', 'knee_extension_peak_angle', 'ad_ab_ratio_asymmetry', 'EDEQ_total', 'rs1800629', 'Cadence_asymmetry_10', 'rs10484958', 'BMD_hip', 'rs1800470', 'rs3045', 'Cadence_asymmetry_12']
Best Average AUC: 0.6124 ± 0.0275
